In [ ]:
# These four input files are not public:
#   speaker_id_singles.json                   - speaker IDs of the single-signer videos (output of speaker_id_age_sex.ipynb:
#                                               a list of {"person_id": ..., "video_ids": [...]})
#   clean_pjm_open_after_dialog_split.csv     - step 02 output joined with the single-signer rows from step 01
#   speaker_identity_embeddings.pkl           - per-video face embeddings of the single-signer videos (video_id -> vector)
#   dialogs_video_embeddings.pkl              - embedding cache for the dialog halves, written by the clustering cell below
# The final result of this step (with speaker_id) is public as splits/pjm_data.csv in
# https://huggingface.co/datasets/croco-corp/pjm-data

In [ ]:
import pandas as pd

data = pd.read_csv('clean_pjm_open_after_dialog_split.csv')
data

In [ ]:
import json

with open('speaker_id_singles.json') as f:
    speaker_ids = json.loads(f.read())

In [ ]:
speaker_ids[:5]

In [ ]:
# Create a dictionary mapping video_id to person_id
video_to_person = {}
for item in speaker_ids:
    person_id = item['person_id']
    for video_id in item['video_ids']:
        video_to_person[video_id] = person_id

# Populate the dataframe with speaker_id
data['speaker_id'] = data['video_id'].map(video_to_person)
data

In [ ]:
non_dialog = data[data['speaker_side'] == '-']
len(non_dialog['speaker_id'].unique())

In [ ]:
import pickle

with open('speaker_identity_embeddings.pkl', 'rb') as f:
    speaker_identity = pickle.load(f)

In [ ]:
import numpy as np

person_identity_centroids = {}
for speaker_id, group_df in non_dialog.groupby(['speaker_id'])['video_id']:
    speaker_id = speaker_id[0]
    ids = group_df.values
    
    embeddings = np.array([speaker_identity[idx] for idx in ids])
    person_identity_centroids[speaker_id] = np.mean(embeddings, axis=0)

In [ ]:
%pip install deepface

In [ ]:
import cv2
import numpy as np
import pandas as pd
from deepface import DeepFace
from sklearn.cluster import AgglomerativeClustering
from tqdm import tqdm
import urllib.request
import os
import pickle

# --- Configuration ---
# Distance threshold for clustering (0.0 to 2.0 for cosine distance)
# Lower = stricter (fewer false positives, maybe more false negatives). 
# 0.3 - 0.4 is a good starting point for DeepFace/VGG-Face.
DISTANCE_THRESHOLD = 0.35 
FRAMES_TO_SAMPLE = 5  # Number of frames to average per video
EMBEDDINGS_FILE = "dialogs_video_embeddings.pkl"

def get_video_embedding(video_url, num_frames=FRAMES_TO_SAMPLE):
    """
    Downloads/streams video, extracts frames, computes face embeddings, 
    and returns the average embedding vector.
    """
    try:
        cap = cv2.VideoCapture(video_url)
        if not cap.isOpened():
            return None

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0: return None
        
        # Sample frames evenly throughout the video
        frame_indices = np.linspace(0, total_frames-1, num_frames, dtype=int)
        
        embeddings = []
        
        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret: continue
            
            try:
                # DeepFace expects BGR (OpenCV default) or RGB. 
                # result is a list of dicts. We take the first face found.
                # enforce_detection=False allows it to skip frames without faces purely.
                result = DeepFace.represent(
                    frame, 
                    model_name="VGG-Face", 
                    enforce_detection=False,
                    detector_backend="opencv" # 'opencv' is fast, 'mtcnn' or 'retinaface' are more accurate but slower
                )
                
                if result:
                    # Take the largest face if multiple are found, or just the first
                    embedding = result[0]["embedding"]
                    embeddings.append(embedding)
            except:
                continue
                
        cap.release()
        
        if not embeddings:
            return None
            
        # Average the embeddings to get a stable representation for the video
        return np.mean(embeddings, axis=0)
    except Exception as e:
        print(f"Error processing {video_url}: {e}")
        return None

# Load existing embeddings if available
dialogs_embeddings_map = {}
if os.path.exists(EMBEDDINGS_FILE):
    print(f"Loading existing embeddings from {EMBEDDINGS_FILE}...")
    with open(EMBEDDINGS_FILE, 'rb') as f:
        dialogs_embeddings_map = pickle.load(f)

working_df = data[data['speaker_side'] != '-'].copy()

for i, rec in tqdm(working_df.iterrows(), total=len(working_df)):
    url = rec['video_url']
    
    # Use cached embedding if available
    if i in dialogs_embeddings_map:
        emb = dialogs_embeddings_map[i]
        continue
    
    emb = get_video_embedding(url)
    if emb is None:
        print(f"Could not extract face from video {video_id}")
        continue 
    
    dialogs_embeddings_map[i] = emb    

# Save Embeddings to Pickle
with open(EMBEDDINGS_FILE, 'wb') as f:
    pickle.dump(dialogs_embeddings_map, f)
print(f"Saved {len(dialogs_embeddings_map)} embeddings to {EMBEDDINGS_FILE}")

In [ ]:
from scipy.spatial.distance import cosine

# 1. Build a registry of known speakers identities (from non-dialogs)
# We assume 'speaker_id' is already populated for non-dialog rows (speaker_side == '-')
known_speakers_df = data[data['speaker_side'] == '-'][['speaker_id', 'age', 'sex']].drop_duplicates()
known_speakers_registry = known_speakers_df.set_index('speaker_id').to_dict('index')

# 2. Iterate and Match
dialog_rows_indices = data[data['speaker_side'] != '-'].index

# Pre-compute demographics for each video_id for efficiency
video_demographics = {}
dialog_video_ids = data[data['speaker_side'] != '-']['video_id'].unique()

print("Pre-computing video demographics...")
for vid in dialog_video_ids:
    video_rows = data[data['video_id'] == vid]
    # Pool all valid ages and sexes found in rows for this video
    valid_ages = set(video_rows['age'].dropna().unique())
    valid_sexes = set(video_rows['sex'].dropna().unique())
    video_demographics[vid] = {'ages': valid_ages, 'sexes': valid_sexes}

matches_count = 0
for idx in tqdm(dialog_rows_indices, desc="Matching speakers"):
    if idx not in dialogs_embeddings_map or dialogs_embeddings_map[idx] is None:
        continue
        
    embedding = dialogs_embeddings_map[idx]
    video_id = data.at[idx, 'video_id']
    
    # Get pooled demographics constraints
    constraints = video_demographics.get(video_id, {'ages': set(), 'sexes': set()})
    
    # Filter candidates: Must match ANY valid age AND ANY valid sex
    candidate_ids = []
    for sp_id, meta in known_speakers_registry.items():
        if meta['age'] in constraints['ages'] and meta['sex'] in constraints['sexes']:
            candidate_ids.append(sp_id)
            
    # Fallback if no demographics match
    if not candidate_ids:
        candidate_ids = list(person_identity_centroids.keys())

    # Find closest centroid among candidates
    best_score = float('inf')
    best_speaker = None
    
    for cand_id in candidate_ids:
        if cand_id not in person_identity_centroids:
            continue
            
        centroid = person_identity_centroids[cand_id]
        dist = cosine(embedding, centroid)
        
        if dist < best_score:
            best_score = dist
            best_speaker = cand_id
            
    if best_speaker:
        data.at[idx, 'speaker_id'] = best_speaker
        matches_count += 1

print(f"Assigned speaker_id to {matches_count} dialog splits.")
data[data['speaker_side'] != '-'][['video_id', 'speaker_side', 'age', 'sex', 'speaker_id']].head(15)

In [ ]:
import cv2
import numpy as np
import math
import matplotlib.pyplot as plt

def visualize_matches(start_idx=0, end_idx=20):
    """
    Visualizes matches for dialog video splits.
    Shows the dialog video frame on the left, and a frame from a known video of the assigned speaker on the right.
    """
    
    dialog_rows = data[data['speaker_side'] != '-']
    # Filter for rows that actually have an assigned speaker_id after our matching process
    matched_dialogs = dialog_rows[dialog_rows['speaker_id'].notna()]
    
    # Slice the range we want to inspect
    subset = matched_dialogs.iloc[start_idx:end_idx]
    
    if len(subset) == 0:
        print("No matches found in the specified range.")
        return

    TILE_SIZE = 300
    HEADER_HEIGHT = 50
    ROWS = len(subset)

    # Canvas dimensions
    canvas_height = ROWS * TILE_SIZE
    canvas_width = 2 * TILE_SIZE # 2 columns: Query | Match
    
    # White canvas
    canvas = np.ones((canvas_height, canvas_width, 3), dtype=np.uint8) * 255
    
    print(f"Generating visualization for {ROWS} matches...")
    
    for i, (dialog_idx, row) in tqdm(enumerate(subset.iterrows()), total=len(subset)):
        y_offset = i * TILE_SIZE
        
        # --- 1. Left: The Dialog Video Frame ---
        dialog_url = row['video_url']
        assigned_id = row['speaker_id']
        video_id = row['video_id']
        side = row['speaker_side']
        
        # Capture frame
        cap = cv2.VideoCapture(dialog_url)
        if cap.isOpened():
            ret, frame = cap.read()
            cap.release()
            if ret:
                frame = cv2.resize(frame, (TILE_SIZE, TILE_SIZE))
                # Add Text
                cv2.putText(frame, f"Dialog: {video_id} ({side})", (10, 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.putText(frame, f"Assigned: {assigned_id}", (10, 60), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                
                canvas[y_offset:y_offset+TILE_SIZE, 0:TILE_SIZE] = frame
        
        # --- 2. Right: Frame from a Known Video of the Assigned Speaker ---
        # Find a non-dialog video for this speaker to compare against
        reference_rows = data[(data['speaker_id'] == assigned_id) & (data['speaker_side'] == '-')]
        
        if not reference_rows.empty:
            # Pick the first one as reference
            ref_row = reference_rows.iloc[0]
            ref_url = ref_row['video_url']
            ref_vid_id = ref_row['video_id']
            
            cap_ref = cv2.VideoCapture(ref_url)
            if cap_ref.isOpened():
                ret, frame_ref = cap_ref.read()
                cap_ref.release()
                if ret:
                    frame_ref = cv2.resize(frame_ref, (TILE_SIZE, TILE_SIZE))
                    
                    cv2.putText(frame_ref, f"Ref Video: {ref_vid_id}", (10, 30), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    cv2.putText(frame_ref, f"ID: {assigned_id}", (10, 60), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                    
                    canvas[y_offset:y_offset+TILE_SIZE, TILE_SIZE:2*TILE_SIZE] = frame_ref
        else:
             # Placeholder text if no reference found
             cv2.putText(canvas, "No Ref Video", (TILE_SIZE + 50, y_offset + 150), 
                         cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # Save and Show (using matplotlib for inline display in notebook)
    output_path = 'match_verification.jpg'
    cv2.imwrite(output_path, canvas)
    print(f"Saved visualization to {output_path}")

# Example usage:
visualize_matches(start_idx=300, end_idx=400)

In [ ]:
corrections = [
    (20, "left", "F_32-45_016"),
    (22, "left", "F_32-45_001"),
    (22, "right", "F_46-60_001"),
    (63, "right", "M_18-30_004"),
    (67, "left", "F_32-45_016"),
    (68, "right", "M_60+_006"),
    (78, "right", "M_18-30_006"),
    (80, "right", "F_46-60_020"),
    (130, "left", "M_32-45_028"),
    (130, "right", "M_32-45_017"),
    (156, "right", "F_60+_007"),
    (144, "right", "M_60+_006"),
    (194, "right", "F_18-30_008"),
    (169, "left", "F_46-60_010"),
    (196, "left", "M_32-45_014"),
    (206, "right", "F_46-60_014"),
    (218, "left", "M_18-30_012"),
    (214, "left", "M_18-30_012"),
    (205, "right", "M_32-45_001"),
    (183, "right", "F_46-60_001"),
    (160, "left", "F_18-30_003"),
    (257, "right", "F_60+_004"),
    (237, "left", "M_32-45_002"),
    (237, "right", "M_32-45_024"),
    (240, "left", "F_32-45_019"),
    (265, "right", "M_32-45_003"),
    (263, "left", "F_46-60_017"),
    (299, "left", "M_32-45_010"),
    (299, "right", "F_18-30_001"),
    (334, "right", "F_32-45_006"),
    (339, "left", "F_46-60_007"),
    (339, "right", "F_46-60_008"),
    (406, "left", "F_32-45_020"),
    (405, "left", "F_18-30_013"),
    (405, "right", "F_60+_004"),
    (427, "right", "F_60+_013"),
    (430, "left", "M_32-45_020"),
    (429, "right", "M_32-45_017"),
    (433, "right", "F_46-60_020"),
    (445, "left", "F_18-30_014"),
    (446, "left", "F_46-60_011"),
    (464, "right", "M_32-45_003"),
    (505, "right", "F_46-60_020"),
    (511, "right", "F_46-60_008"),
    (512, "left", "M_32-45_028"),
    (531, "right", "M_32-45_024"),
    (535, "left", "F_18-30_003"),
    (548, "left", "M_60+_016"),
    (475, "right", "F_18-30_008"),
    (482, "left", "M_60+_002"),
    (482, "right", "M_60+_009"),
    (595, "right", "F_18-30_009"),
    (583, "right", "F_60+_008"),
    (577, "right", "F_46-60_022"),
    (612, "left", "M_32-45_006"),
    (612, "right", "F_32-45_025"),
    (618, "right", "M_32-45_021"),
    (630, "left", "M_18-30_012"),
    (613, "right", "F_32-45_015"),
    (685, "left", "F_32-45_023"),
    (675, "right", "F_60+_013"),
    (683, "right", "F_60+_008"),
    (725, "right", "M_46-60_003"),
    (708, "right", "M_46-60_002"),
    (688, "right", "M_18-30_004"),
]

print(f"Applying {len(corrections)} manual corrections...")

for video_id, side, new_speaker_id in corrections:
    # Find the matching row
    mask = (data['video_id'] == video_id) & (data['speaker_side'] == side)
    
    if mask.sum() == 0:
        print(f"Warning: No Match found for Video {video_id} ({side})")
        continue

    # Update
    data.loc[mask, 'speaker_id'] = new_speaker_id

print("Corrections applied.")
data[data['speaker_side'] != '-'].sample(10)

In [ ]:
# 1. Build Ground Truth from Single Videos (Non-Dialogs)
# These rows have trusted age/sex for each speaker_id
ground_truth_df = data[data['speaker_side'] == '-'][['speaker_id', 'age', 'sex']].drop_duplicates()
ground_truth_map = ground_truth_df.set_index('speaker_id').to_dict('index')

print(f"Loaded demographics for {len(ground_truth_map)} speakers from single videos.")

# 2. Update Dialog Records
# Target rows: Dialog splits that have an assigned speaker_id
dialog_mask = (data['speaker_side'] != '-') & (data['speaker_id'].notna())
dialog_indices = data[dialog_mask].index

updated_count = 0
for idx in dialog_indices:
    sp_id = data.at[idx, 'speaker_id']
    
    # If we have ground truth for this speaker, overwrite age/sex in the dialog row
    if sp_id in ground_truth_map:
        truth = ground_truth_map[sp_id]
        data.at[idx, 'age'] = truth['age']
        data.at[idx, 'sex'] = truth['sex']
        updated_count += 1
    else:
        # This might happen if a speaker_id was assigned manually but doesn't exist in the singles dataset 
        # (e.g. if we created a new ID for a dialog-only speaker, though current logic assumes we matched to existing ones)
        pass

print(f"Updated age and sex for {updated_count} dialog records.")

# Verify results
data[data['speaker_side'] != '-'][['video_id', 'speaker_side', 'speaker_id', 'age', 'sex']].sample(10)

In [ ]:
data.to_csv("data_21-03-2026.csv", index=False)

In [ ]:
# Colab only: we mounted Google Drive here to keep the result between sessions. Not needed elsewhere.

In [ ]:
# data_21-03-2026.csv written above is the table with speaker_id for all rows; copy it somewhere permanent.

In [ ]:
data[data['video_id'] == 683][['speaker_id', 'speaker_side']]
# len(data[(data['age'] == '60+') & (data['sex'] == 'F') & (data['speaker_side'] == '-')])

In [ ]:
data[data['speaker_id'] == "M_60+_002"][['video_id', 'speaker_side']]